In [10]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// Parameters
#define BM 128
#define BN 128
#define BK 16
#define TM 8
#define TN 8

// Dynamic calculation of threads per block based on the tile dimensions
#define NUM_THREADS ((BN / TN) * (BM / TM))

// ---- KERNEL CUDA ----
__global__ void sgemm_float4_only(int pad_n, const float *A, const float *B, float *C) {

    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int tid = ty * (BN / TN) + tx;

    float res[TM][TN] = {0.0f};
    float a_reg[TM];
    float b_reg[TN];

    // Change: loading on the Shared Memory slices through the specific instruction float4
    for (int p = 0; p < (pad_n + BK - 1) / BK; ++p) {

        // Tile A
        for (int i = 0; i < (BM * BK) / (NUM_THREADS * 4); ++i) {
            int load_idx = i * NUM_THREADS + tid;
            int a_row = load_idx / (BK / 4);
            int a_col_v = (load_idx % (BK / 4)) * 4;
            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col_v;

            // Vectorized Memory Access: 16 byte through only one 128-bit transiction
            float4 tmpA = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
            if (g_a_row < pad_n && g_a_col < pad_n) {
                tmpA = *(const float4*)&A[g_a_row * pad_n + g_a_col];
            }
            // Vectorized Write in Shared Memory
            *(float4*)&sA[a_row][a_col_v] = tmpA;
        }

        // Tile B
        for (int i = 0; i < (BK * BN) / (NUM_THREADS * 4); ++i) {
            int load_idx = i * NUM_THREADS + tid;
            int b_row = load_idx / (BN / 4);
            int b_col_v = (load_idx % (BN / 4)) * 4;
            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col_v;

            // Vectorized Memory Access
            float4 tmpB = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
            if (g_b_row < pad_n && g_b_col < pad_n) {
                tmpB = *(const float4*)&B[g_b_row * pad_n + g_b_col];
            }
            *(float4*)&sB[b_row][b_col_v] = tmpB;
        }

        __syncthreads();

        // Computation on Shared Memory and Registers
        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }


    // Saving in the Global Memory C
    int row = by * BM + ty * TM;
    int col = bx * BN + tx * TN;

    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row + i;
            int c_col = col + j;
            if (c_row < pad_n && c_col < pad_n) {
                C[c_row * pad_n + c_col] = res[i][j];
            }
        }
    }
}


// ---- HOST ----
int main(int argc, char **argv) {

    if (argc < 2) {
        fprintf(stderr, "Error: missing argument in %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);

    // Change: addition of padding logic to handle float4 instructions
    int pad_n = (n + BM - 1) / BM * BM;

    // Host Memory Allocation
    size_t pad_bytes = (size_t)pad_n * pad_n * sizeof(float);
    float *h_a = (float*)malloc(pad_bytes);
    float *h_b = (float*)malloc(pad_bytes);
    float *h_c = (float*)malloc(pad_bytes);

    // Initialization through padding
    for (int i = 0; i < pad_n; i++) {
        for (int j = 0; j < pad_n; j++) {
            if (i < n && j < n) {
                h_a[i * pad_n + j] = 2.0f;
                h_b[i * pad_n + j] = 3.0f;
            } else {
                h_a[i * pad_n + j] = 0.0f;
                h_b[i * pad_n + j] = 0.0f;
            }
            h_c[i * pad_n + j] = 0.0f;
        }
    }

    // Device Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, pad_bytes);
    cudaMalloc(&d_b, pad_bytes);
    cudaMalloc(&d_c, pad_bytes);

    // Copy from Host to Device
    cudaMemcpy(d_a, h_a, pad_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, pad_bytes, cudaMemcpyHostToDevice);

    // Execution configuration for the Kernel launch: Grid and Block dimensions
    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((pad_n + BN - 1) / BN, (pad_n + BM - 1) / BM);

    // CUDA Event for Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Kernel Launch + Timing
    cudaEventRecord(start);
    sgemm_float4_only<<<blocksPerGrid, threadsPerBlock>>>(pad_n, d_a, d_b, d_c);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    // Copy from Device to Host
    cudaMemcpy(h_c, d_c, pad_bytes, cudaMemcpyDeviceToHost);

    // Results
    FILE *f = fopen("mat-res-float4.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * pad_n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_float4.cu', 'w') as f:
    f.write(cell_str)

In [11]:
!nvcc -arch=sm_75 -O3 cuda_float4.cu -o cuda_float4

In [12]:
!nvprof ./cuda_float4 20000

==3612== NVPROF is profiling process 3612, command: ./cuda_float4 20000
Execution Time: 3.7156 seconds
==3612== Profiling application: ./cuda_float4 20000
==3612== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   78.13%  3.71525s         1  3.71525s  3.71525s  3.71525s  sgemm_float4_only(int, float const *, float const *, float*)
                   14.49%  688.93ms         2  344.46ms  341.03ms  347.90ms  [CUDA memcpy HtoD]
                    7.38%  350.80ms         1  350.80ms  350.80ms  350.80ms  [CUDA memcpy DtoH]
      API calls:   75.07%  3.71526s         1  3.71526s  3.71526s  3.71526s  cudaEventSynchronize
                   21.02%  1.04055s         3  346.85ms  341.26ms  351.17ms  cudaMemcpy
                    3.65%  180.50ms         3  60.167ms  153.24us  180.18ms  cudaMalloc
                    0.20%  9.9838ms         3  3.3279ms  2.9631ms  3.8704ms  cudaFree
                    0.05%  2.5357ms       114 